# Etapa 1) Extração dos Dados

## Fonte de Dados
- Arquivo: `clientes_bancarios.csv`
- Encoding: UTF-8
- Tipo: CSV (Comma Separated Values)

In [ ]:
import pandas as pd
import glob
import os

def carregar_csvs_para_dataframe(caminho_pasta, header):
    """
    Lê múltiplos arquivos CSV de uma pasta especificada e os combina num único DataFrame.
    """
    # Usa o glob para encontrar todos os ficheiros com extensão .csv na pasta
    padrao_busca = os.path.join(caminho_pasta, "*.csv")
    arquivos_csv = glob.glob(padrao_busca)
    
    # Verifica se encontrou algum ficheiro
    if not arquivos_csv:
        print(f"Aviso: Nenhum ficheiro CSV encontrado na pasta '{caminho_pasta}'.")
        return pd.DataFrame() # Retorna um DataFrame vazio
    
    print(f"Foram encontrados {len(arquivos_csv)} ficheiro(s) CSV. A iniciar a leitura...\n")
    
    lista_dataframes = []
    
    # Itera sobre cada ficheiro encontrado
    for arquivo in arquivos_csv:
        try:
            # Lê o ficheiro CSV
            # encoding='latin1' resolve o erro de descodificação (caracteres especiais como ç, á, ã)
            # sep=',' é usado conforme especificado (separado por vírgula)
            df_temporario = pd.read_csv(arquivo, encoding='latin1', sep=';', header= header)
            
            # Opcional: Adiciona uma coluna para rastrear de qual ficheiro a linha veio
            df_temporario['arquivo_origem'] = os.path.basename(arquivo)
            
            lista_dataframes.append(df_temporario)
            print(f"Sucesso: {os.path.basename(arquivo)} lido com {len(df_temporario)} linhas.")
            
        except Exception as e:
            print(f"Erro ao ler o ficheiro {os.path.basename(arquivo)}: {e}")
    
    # Se a lista não estiver vazia, concatena todos os DataFrames
    if lista_dataframes:
        # ignore_index=True garante que o índice do DataFrame final seja sequencial
        df_final = pd.concat(lista_dataframes, ignore_index=True)
        print("\nTodos os ficheiros foram combinados com sucesso!")
        print(f"Total de linhas no DataFrame final: {len(df_final)}")
        return df_final
    else:
        print("\nFalha ao extrair dados dos ficheiros.")
        return pd.DataFrame()

# ==========================================
# Execução do script
# ==========================================
if __name__ == "__main__":
    # Caminho especificado
    pasta_origem_detran = r"C:/Users/Natan/Downloads/banco-digital-MKZ-1/app/Detran/origem/detran"
    pasta_origem_dnit = r"C:/Users/Natan/Downloads/banco-digital-MKZ-1/app/Detran/origem/dnit"
    
    # Chama a função e guarda o resultado na variável 'df'
    df_detran = carregar_csvs_para_dataframe(pasta_origem_detran,0)
    df_dnit = carregar_csvs_para_dataframe(pasta_origem_dnit, 0)
    
    # Exibe as 5 primeiras linhas do DataFrame resultante (se não estiver vazio)
    if not df_detran.empty:
        print("\nVisualização dos primeiros registos do DETRAN:")
        print(df_detran.head())
    if not df_dnit.empty:
        print("\nVisualização dos primeiros registos do DNIT:")
        print(df_dnit.head())

# Etapa 2) Preparação dos dados

## Tratamento Completo de Tipos
Conversão e validação de todos os campos:
- Inteiros: idade, score_credito, numero_transacoes_mes
- Decimal(16,2): renda_mensal, saldo_conta
- Strings (categóricas): genero, possui_cartao_credito, tipo_conta, usa_app_mobile, participa_programa_fidelidade, churn

In [ ]:
print(df_dnit.head(5))

In [ ]:

print(df_detran.head(5))

In [ ]:
print(df_dnit.columns.tolist())

In [ ]:
print(df_detran.columns.tolist())

In [ ]:
import unicodedata
import re

def normalizar_colunas(df):

    # 1️⃣ remover colunas lixo do CSV
    df = df.loc[:, ~df.columns.str.contains('^Unnamed', case=False)]

    # 2️⃣ corrigir encoding quebrado (latin1 → utf8)
    df.columns = (
        df.columns
        .str.encode('latin1', errors='ignore')
        .str.decode('utf-8', errors='ignore')
    )

    # 3️⃣ remover BOM (ï»¿)
    df.columns = df.columns.str.replace('ï»¿', '', regex=False)

    # 4️⃣ função de normalização padrão DW
    def normalize(col):

        # remover acentos
        col = unicodedata.normalize('NFKD', col)\
            .encode('ascii', 'ignore')\
            .decode('utf-8')

        # lowercase
        col = col.lower()

        # espaços → _
        col = col.replace(' ', '_')

        # remove caracteres especiais
        col = re.sub(r'[^a-z0-9_]', '', col)

        # remove múltiplos _
        col = re.sub(r'_+', '_', col)

        return col.strip('_')

    df.columns = [normalize(c) for c in df.columns]

    return df

In [ ]:
df_dnit = normalizar_colunas(df_dnit)
print(df_dnit.columns.tolist())

In [ ]:
df_detran = normalizar_colunas(df_detran)
print(df_detran.columns.tolist())

In [ ]:
def select_columns(df, columns, verbose=True):
    """
    Mantém apenas as colunas desejadas em um DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame de entrada
    columns : list
        Lista de colunas desejadas (schema alvo)
    verbose : bool
        Mostra aviso de colunas faltantes

    Returns
    -------
    pandas.DataFrame
        DataFrame apenas com as colunas selecionadas
    """

    # colunas existentes
    existing_cols = [c for c in columns if c in df.columns]

    # colunas faltantes
    missing_cols = list(set(columns) - set(df.columns))

    if verbose and missing_cols:
        print(f"⚠️ Colunas não encontradas: {missing_cols}")

    # mantém ordem do schema
    df = df.loc[:, existing_cols]

    return df

In [ ]:
schema_dnit = [
    'uf',
    'rodovia',
    'km_inicial',
    'km_final',
    'extensao_km',
    'data',
    'latitude',
    'longitude',
    'icmnp',
    'arquivo_origem',
    'observacao',
    'icc',
    'icp',
    'icm'
]

df_dnit = select_columns(df_dnit, schema_dnit)
print(df_dnit.columns.tolist())

In [ ]:
SCHEMA_DETRAN = [
    'id',
    'data_inversa',
    'dia_semana',
    'horario',
    'uf',
    'br',
    'km',
    'municipio',
    'causa_principal',
    'causa_acidente',
    'ordem_tipo_acidente',
    'tipo_acidente',
    'classificacao_acidente',
    'fase_dia',
    'sentido_via',
    'condicao_metereologica',
    'tipo_pista',
    'tracado_via',
    'uso_solo',
    'id_veiculo',
    'tipo_veiculo',
    'marca',
    'ano_fabricacao_veiculo',
    'estado_fisico',
    'idade',
    'sexo',
    'ilesos',
    'feridos_leves',
    'feridos_graves',
    'mortos',
    'latitude',
    'longitude'
]
df_detran = select_columns(df_detran, SCHEMA_DETRAN)
print(df_detran.columns.tolist())

In [ ]:
def limpar_dados(df):
    """
    Remove linhas vazias e duplicadas do DataFrame.
    Considera uma linha vazia se todas as colunas originais do CSV forem nulas.
    """
    print("\n--- A iniciar limpeza de dados ---")
    
    linhas_iniciais = len(df)
    
    # Define as colunas originais (ignora 'arquivo_origem' para a verificação de linhas vazias)
    colunas_originais = [col for col in df.columns if col != 'arquivo_origem']
    
    # 1. Remove as linhas onde TODAS as colunas originais são NaN (vazias)
    df_limpo = df.dropna(how='all', subset=colunas_originais)
    linhas_apos_vazias = len(df_limpo)
    print(f"Linhas vazias removidas: {linhas_iniciais - linhas_apos_vazias}")
    
    # 2. Remove as linhas duplicadas (mantém a primeira ocorrência)
    df_limpo = df_limpo.drop_duplicates()
    linhas_apos_duplicadas = len(df_limpo)
    print(f"Linhas duplicadas removidas: {linhas_apos_vazias - linhas_apos_duplicadas}")
    
    print(f"Total de linhas finais (após limpeza): {len(df_limpo)}")
    print("----------------------------------")
    
    return df_limpo

# Se o DataFrame não estiver vazio, limpa e exibe os dados
if not df_detran.empty:
    # Chama a nova função de limpeza
    df_detran_limpo = limpar_dados(df_detran)
    
    print("\nVisualização dos primeiros registos:")
    # print(df_detran_limpo.head())

# Se o DataFrame não estiver vazio, limpa e exibe os dados
if not df_dnit.empty:
    # Chama a nova função de limpeza
    df_dnit_limpo = limpar_dados(df_dnit)
    
    print("\nVisualização dos primeiros registos:")
    # print(df_dnit_limpo.head())

In [ ]:
print(df_detran_limpo.head())

In [ ]:
print(df_dnit_limpo.head())

In [ ]:
import pandas as pd

def padronizar_campos_join(df, coluna_data, lat_col="latitude", lon_col="longitude"):
    """
    Padroniza campos utilizados como chave de join:
    - converte datas para datetime
    - corrige latitude/longitude (vírgula decimal)
    - converte coordenadas para float
    - arredonda coordenadas para evitar problemas de precisão
    """

    # =========================
    # 1️⃣ Padronizar DATA
    # =========================
    df[coluna_data] = pd.to_datetime(
        df[coluna_data],
        errors="coerce",
        dayfirst=True  # funciona para 03/06/2022
    )

    # =========================
    # 2️⃣ Padronizar COORDENADAS
    # =========================
    for col in [lat_col, lon_col]:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)  # decimal BR → padrão
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

        # arredondamento (ESSENCIAL para join geográfico)
        df[col] = df[col].round(5)

    return df

In [ ]:
df_detran_limpo = padronizar_campos_join(
    df_detran_limpo,
    coluna_data="data_inversa"
)

In [ ]:
df_dnit_limpo = padronizar_campos_join(
    df_dnit_limpo,
    coluna_data="data"
)

In [ ]:
print("===== CHAVES DF_DETRAN =====")
print(df_detran_limpo[["data_inversa", "latitude", "longitude"]].head(5))

print("\n===== CHAVES DF_DNIT =====")
print(df_dnit_limpo[["data", "latitude", "longitude"]].head(5))

In [ ]:
import pandas as pd

def left_join_detran_dnit(df_detran, df_dnit, verbose=True):
    """
    Realiza LEFT JOIN entre DETRAN e DNIT usando:
    latitude, longitude e data.

    Retorna dataframe enriquecido.
    """

    df_detran = df_detran.copy()
    df_dnit = df_dnit.copy()

    # -----------------------------
    # 1️⃣ validar colunas
    # -----------------------------
    keys_detran = ['latitude', 'longitude', 'data_inversa']
    keys_dnit = ['latitude', 'longitude', 'data']

    for c in keys_detran:
        if c not in df_detran.columns:
            raise KeyError(f"Coluna ausente no DETRAN: {c}")

    for c in keys_dnit:
        if c not in df_dnit.columns:
            raise KeyError(f"Coluna ausente no DNIT: {c}")

    # -----------------------------
    # 2️⃣ garantir tipos corretos
    # -----------------------------
    for df in [df_detran, df_dnit]:
        df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce').round(5)
        df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce').round(5)

    df_detran['data_inversa'] = pd.to_datetime(df_detran['data_inversa'], errors='coerce')
    df_dnit['data'] = pd.to_datetime(df_dnit['data'], errors='coerce')

    # -----------------------------
    # 3️⃣ executar LEFT JOIN
    # -----------------------------
    df_joined = df_detran.merge(
        df_dnit,
        left_on=['latitude', 'longitude', 'data_inversa'],
        right_on=['latitude', 'longitude', 'data'],
        how='left',
        suffixes=('', '_dnit')
    )

    # -----------------------------
    # 4️⃣ log opcional
    # -----------------------------
    if verbose:
        print(f"✅ LEFT JOIN concluído")
        print(f"Linhas DETRAN: {len(df_detran)}")
        print(f"Linhas resultado: {len(df_joined)}")

        match_rate = df_joined['data'].notna().mean() * 100
        print(f"Taxa de match: {match_rate:.2f}%")

    return df_joined

In [ ]:
df_detran_dnit_joined = left_join_detran_dnit(
    df_detran_limpo,
    df_dnit_limpo
)

In [ ]:
import os

def salvar_csv(df, caminho, nome_arquivo, separador=";"):
    """
    Salva um DataFrame em CSV.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame a ser salvo
    caminho : str
        Diretório destino
    nome_arquivo : str
        Nome do arquivo CSV
    separador : str
        Separador do CSV (default=';')
    """

    # cria pasta se não existir
    os.makedirs(caminho, exist_ok=True)

    # caminho completo
    caminho_completo = os.path.join(caminho, nome_arquivo)

    # salvar arquivo
    df.to_csv(
        caminho_completo,
        sep=separador,
        index=False,
        encoding="utf-8"
    )

    print(f"✅ Arquivo salvo com sucesso em:\n{caminho_completo}")

In [ ]:
salvar_csv(
    df_detran_dnit_joined,
    caminho="C:/Users/Natan/Downloads/banco-digital-MKZ-1/app/Detran/destino/sot",
    nome_arquivo="sot_detran_denit.csv"
)